In [0]:
def test_duolingo_streak(spark):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, row_number, to_date, datediff, unix_date

    data = [
        ("U1", "2024-01-01"),
        ("U1", "2024-01-02"),
        ("U1", "2024-01-03")
    ]

    columns = ["user_id", "activity_date"]
    df = spark.createDataFrame(data, columns) \
              .withColumn("activity_date", to_date("activity_date"))

    df = df.dropDuplicates(["user_id", "activity_date"])

    window_spec = Window.partitionBy("user_id").orderBy("activity_date")

    df = df.withColumn("rn", row_number().over(window_spec)) \
           .withColumn("group_key", unix_date(col("activity_date")) - col("rn"))

    result_df = df.groupBy("user_id", "group_key") \
                  .count() \
                  .filter(col("count") >= 3)

    result = {r["user_id"] for r in result_df.collect()}

    expected = {"U1"}

    try:
        assert result == expected
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")
        print("Expected:", expected)
        print("Got:", result)

test_duolingo_streak(spark)